In [ ]:
# this may be example of my sfdps algorithm failing.

# Search SFDPS logs for `a003a8` on `2026-03-01`

This notebook searches the local derived SFDPS flights, algorithm output, SFDPS intermediate parquet logs, and raw gzipped SFDPS logs for the target ICAO. It also carries forward related identifiers found in derived data, such as registration and callsign, so later cells can catch raw log rows that do not include the ICAO address directly.

In [1]:
from __future__ import annotations

import gzip
import os
import re
import sys
from datetime import date
from pathlib import Path

import polars as pl

TARGET_DATE = date(2026, 3, 1)
TARGET_ICAO = "a003a8"
INITIAL_RELATED_VALUES = {TARGET_ICAO, "A003A8", "N10PH"}

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / "src"))

def day_partition(d: date) -> str:
    return f"year={d.year}/month={d.month:02d}/day={d.day:02d}"

PARTITION = day_partition(TARGET_DATE)
print(REPO_ROOT)
print(PARTITION)

/Users/jonahgoode/Documents/PlaneQuery/Code/planequery-flights-algorithm
year=2026/month=03/day=01


In [2]:
def ordered_unique(paths: list[Path]) -> list[Path]:
    seen = set()
    out = []
    for path in paths:
        resolved = path.expanduser().resolve()
        if resolved not in seen:
            seen.add(resolved)
            out.append(resolved)
    return out

output_dir = Path(os.getenv("OUTPUT_DIR", REPO_ROOT)).expanduser()
candidate_roots = ordered_unique([
    output_dir,
    REPO_ROOT,
    REPO_ROOT.parent / "planequery-data",
])

derived_sfdps_paths = ordered_unique([
    root / "data/flights/sfdps/v1" / PARTITION / "part-0.parquet"
    for root in candidate_roots
])
algorithm_paths = ordered_unique([
    root / "data/flights/algorithm/v1/adsblol" / PARTITION / "flights.parquet"
    for root in candidate_roots
])
intermediate_sfdps_paths = ordered_unique([
    root / "data/intermediate/sfdps-logs" / PARTITION / f"sfdps-logs_{TARGET_DATE:%Y_%m_%d}.parquet"
    for root in candidate_roots
])
raw_sfdps_dirs = ordered_unique([
    root / "data/raw/sfdps-logs" / PARTITION
    for root in candidate_roots
])

for label, paths in {
    "derived SFDPS flights": derived_sfdps_paths,
    "algorithm flights": algorithm_paths,
    "intermediate SFDPS parquet": intermediate_sfdps_paths,
    "raw SFDPS dirs": raw_sfdps_dirs,
}.items():
    print(f"\n{label}")
    for path in paths:
        print("exists" if path.exists() else "missing", path)


derived SFDPS flights
exists /Volumes/T2-SSD/planequery/data/flights/sfdps/v1/year=2026/month=03/day=01/part-0.parquet
exists /Users/jonahgoode/Documents/PlaneQuery/Code/planequery-flights-algorithm/data/flights/sfdps/v1/year=2026/month=03/day=01/part-0.parquet
exists /Users/jonahgoode/Documents/PlaneQuery/Code/planequery-data/data/flights/sfdps/v1/year=2026/month=03/day=01/part-0.parquet

algorithm flights
exists /Volumes/T2-SSD/planequery/data/flights/algorithm/v1/adsblol/year=2026/month=03/day=01/flights.parquet
exists /Users/jonahgoode/Documents/PlaneQuery/Code/planequery-flights-algorithm/data/flights/algorithm/v1/adsblol/year=2026/month=03/day=01/flights.parquet
exists /Users/jonahgoode/Documents/PlaneQuery/Code/planequery-data/data/flights/algorithm/v1/adsblol/year=2026/month=03/day=01/flights.parquet

intermediate SFDPS parquet
exists /Volumes/T2-SSD/planequery/data/intermediate/sfdps-logs/year=2026/month=03/day=01/sfdps-logs_2026_03_01.parquet
missing /Users/jonahgoode/Docume

In [3]:
def contains_any_expr(columns: list[str], values: set[str]) -> pl.Expr:
    escaped = [re.escape(v) for v in values if v]
    if not escaped:
        return pl.lit(False)
    pattern = "(?i)(" + "|".join(sorted(escaped, key=len, reverse=True)) + ")"
    return pl.any_horizontal([
        pl.col(col).cast(pl.String, strict=False).str.contains(pattern).fill_null(False)
        for col in columns
    ])

def read_matches(path: Path, values: set[str], columns: list[str] | None = None) -> pl.DataFrame:
    df = pl.read_parquet(path)
    search_columns = columns or df.columns
    search_columns = [col for col in search_columns if col in df.columns]
    if not search_columns:
        return df.clear()
    return df.filter(contains_any_expr(search_columns, values))

def display_matches(label: str, path: Path, values: set[str], columns: list[str] | None = None) -> pl.DataFrame:
    print(f"\n## {label}")
    print(path)
    if not path.exists():
        print("missing")
        return pl.DataFrame()
    matches = read_matches(path, values, columns=columns)
    print(f"rows: {matches.height}")
    if matches.height:
        display(matches)
    return matches

In [4]:
related_values = set(INITIAL_RELATED_VALUES)

derived_matches = []
for path in derived_sfdps_paths:
    matches = display_matches(
        "derived SFDPS flights",
        path,
        related_values,
        columns=["icao", "registration", "callsign"],
    )
    if matches.height:
        derived_matches.append(matches)
        for col in ["icao", "registration", "callsign"]:
            if col in matches.columns:
                related_values.update(v for v in matches[col].drop_nulls().cast(pl.String).unique().to_list() if v)

print("related values for raw/intermediate searches:", sorted(related_values))


## derived SFDPS flights
/Volumes/T2-SSD/planequery/data/flights/sfdps/v1/year=2026/month=03/day=01/part-0.parquet
rows: 2


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a003a8""","""TWY116""","""N10PH""",2026-03-01 18:08:00,"""KBCT""",2026-03-01 18:59:00,"""MYEN""",false,true,false,false,"""C56X""","""BLUE SKY HARBOUR LLC""","""CESSNA 560XL Citation XLS""",""""""
"""a003a8""","""TWY116""","""N10PH""",2026-03-01 19:54:00,"""MYEN""",2026-03-01 20:52:00,"""KFXE""",false,true,false,false,"""C56X""","""BLUE SKY HARBOUR LLC""","""CESSNA 560XL Citation XLS""",""""""



## derived SFDPS flights
/Users/jonahgoode/Documents/PlaneQuery/Code/planequery-flights-algorithm/data/flights/sfdps/v1/year=2026/month=03/day=01/part-0.parquet
rows: 2


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a003a8""","""TWY116""","""N10PH""",2026-03-01 18:08:00,"""KBCT""",2026-03-01 18:59:00,"""MYEN""",false,true,false,false,"""C56X""","""BLUE SKY HARBOUR LLC""","""CESSNA 560XL Citation XLS""",""""""
"""a003a8""","""TWY116""","""N10PH""",2026-03-01 19:54:00,"""MYEN""",2026-03-01 20:52:00,"""KFXE""",false,true,false,false,"""C56X""","""BLUE SKY HARBOUR LLC""","""CESSNA 560XL Citation XLS""",""""""



## derived SFDPS flights
/Users/jonahgoode/Documents/PlaneQuery/Code/planequery-data/data/flights/sfdps/v1/year=2026/month=03/day=01/part-0.parquet
rows: 2


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category
str,str,str,datetime[ms],str,datetime[ms],str,bool,bool,bool,bool,str,str,str,str
"""a003a8""","""TWY116""","""N10PH""",2026-03-01 18:08:00,"""KBCT""",2026-03-01 18:59:00,"""MYEN""",false,true,false,false,"""C56X""","""BLUE SKY HARBOUR LLC""","""CESSNA 560XL Citation XLS""",""""""
"""a003a8""","""TWY116""","""N10PH""",2026-03-01 19:54:00,"""MYEN""",2026-03-01 20:52:00,"""KFXE""",false,true,false,false,"""C56X""","""BLUE SKY HARBOUR LLC""","""CESSNA 560XL Citation XLS""",""""""


related values for raw/intermediate searches: ['A003A8', 'N10PH', 'TWY116', 'a003a8']


In [5]:
algorithm_matches = []
for path in algorithm_paths:
    matches = display_matches(
        "algorithm flights",
        path,
        related_values,
        columns=["icao", "registration", "callsign", "flight_id"],
    )
    if matches.height:
        algorithm_matches.append(matches)


## algorithm flights
/Volumes/T2-SSD/planequery/data/flights/algorithm/v1/adsblol/year=2026/month=03/day=01/flights.parquet
rows: 1


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,first_message_time,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,flight_id,takeoff_airport_score,landing_airport_score
str,str,str,datetime[ms],str,datetime[ms],str,datetime[ms],f64,f64,i64,i64,datetime[ms],f64,f64,i64,i64,bool,bool,bool,bool,str,str,str,str,str,f64,f64
"""a003a8""","""""","""""",2026-03-01 19:44:39.670,"""MYAN""",2026-03-01 20:52:15.660,"""KFXE""",2026-03-01 20:11:51.100,25.472717,-78.507233,22000,null,2026-03-01 20:52:15.660,26.197403,-80.182938,-25,null,false,true,false,false,"""C56X""","""BLUE SKY HARBOUR LLC""","""CESSNA 560XL Citation XLS""","""A2""","""a003a8_2026-03-01_19-44""",0.4522,0.999969



## algorithm flights
/Users/jonahgoode/Documents/PlaneQuery/Code/planequery-flights-algorithm/data/flights/algorithm/v1/adsblol/year=2026/month=03/day=01/flights.parquet
rows: 1


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,first_message_time,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,flight_id,takeoff_airport_score,landing_airport_score
str,str,str,datetime[ms],str,datetime[ms],str,datetime[ms],f64,f64,i64,i64,datetime[ms],f64,f64,i64,i64,bool,bool,bool,bool,str,str,str,str,str,f64,f64
"""a003a8""","""""","""""",2026-03-01 19:44:39.670,"""MYAF""",2026-03-01 20:52:15.660,"""KFXE""",2026-03-01 20:11:51.100,25.472717,-78.507233,22000,null,2026-03-01 20:52:15.660,26.197403,-80.182938,-25,null,false,true,false,false,"""C56X""","""BLUE SKY HARBOUR LLC""","""CESSNA 560XL Citation XLS""","""A2""","""a003a8_2026-03-01_19-44""",0.107034,0.999966



## algorithm flights
/Users/jonahgoode/Documents/PlaneQuery/Code/planequery-data/data/flights/algorithm/v1/adsblol/year=2026/month=03/day=01/flights.parquet
rows: 1


icao,callsign,registration,takeoff_time,takeoff_airport_ident,landing_time,landing_airport_ident,first_message_time,first_lat,first_lon,first_baro_altitude_ft,first_geom_altitude_ft,last_message_time,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,flight_id,takeoff_airport_score,landing_airport_score
str,str,str,datetime[ms],str,datetime[ms],str,datetime[ms],f64,f64,i64,i64,datetime[ms],f64,f64,i64,i64,bool,bool,bool,bool,str,str,str,str,str,f64,f64
"""a003a8""","""""","""""",2026-03-01 19:44:39.670,"""MYAF""",2026-03-01 20:52:15.660,"""KFXE""",2026-03-01 20:11:51.100,25.472717,-78.507233,22000,null,2026-03-01 20:52:15.660,26.197403,-80.182938,-25,null,false,true,false,false,"""C56X""","""BLUE SKY HARBOUR LLC""","""CESSNA 560XL Citation XLS""","""A2""","""a003a8_2026-03-01_19-44""",0.107034,0.999966


In [6]:
intermediate_matches = []
for path in intermediate_sfdps_paths:
    if not path.exists():
        print("missing", path)
        continue

    schema = pl.read_parquet_schema(path)
    print(f"\nintermediate columns in {path.name}: {len(schema)}")
    likely_id_cols = [
        col for col in schema
        if any(token in col.lower() for token in ["aircraft", "registration", "ident", "address", "gufi", "callsign"])
    ]
    print("likely id columns:", likely_id_cols)

    matches = display_matches(
        "intermediate SFDPS parquet, likely id columns",
        path,
        related_values,
        columns=likely_id_cols,
    )
    if matches.height:
        intermediate_matches.append(matches)

    # Second pass over all columns, slower but useful when the value is buried in an unexpected field.
    all_col_matches = display_matches("intermediate SFDPS parquet, all columns", path, related_values)
    if all_col_matches.height:
        intermediate_matches.append(all_col_matches)


intermediate columns in sfdps-logs_2026_03_01.parquet: 563
likely id columns: ['aircraft_identification', 'gufi', 'sup_fdps_gufi', 'aircraft_address', 'aircraft_registration', 'aircraft_type', 'aircraftDescription.capabilities.communication.otherDataLinkCapabilities', 'aircraftDescription.capabilities.communication.selectiveCallingCode', 'aircraftDescription.capabilities.communication.communicationCode', 'aircraftDescription.capabilities.communication.dataLinkCode', 'aircraftDescription.capabilities.navigation.otherNavigationCapabilities', 'aircraftDescription.capabilities.navigation.navigationCode', 'aircraftDescription.capabilities.navigation.performanceBasedCode', 'aircraftDescription.capabilities.surveillance.otherSurveillanceCapabilities', 'aircraftDescription.capabilities.surveillance.surveillanceCode', 'aircraftDescription.accuracy.cmsFieldType[0].phase', 'aircraftDescription.accuracy.cmsFieldType[0]', 'aircraftDescription.accuracy.cmsFieldType[1].phase', 'aircraftDescription.a

centre,source,system,timestamp,arrival_point,departure_point,position_speed_knots,position_latlon,computer_id,aircraft_identification,flight_status,gufi,flight_plan_id,sup_msg_seq_no,sup_fdps_gufi,sup_flight_plan_seq_no,sup_source_time_and_seq,sup_source_time,agreed.route.expandedRoute.routePoint[0].estimatedTime,agreed.route.expandedRoute.routePoint[0].point.fix,agreed.route.expandedRoute.routePoint[0].point.distance,agreed.route.expandedRoute.routePoint[0].point.radial,agreed.route.expandedRoute.routePoint[1].estimatedTime,agreed.route.expandedRoute.routePoint[1].point.fix,agreed.route.expandedRoute.routePoint[2].estimatedTime,agreed.route.expandedRoute.routePoint[2].point.fix,agreed.route.expandedRoute.routePoint[3].estimatedTime,agreed.route.expandedRoute.routePoint[3].point.fix,arrival_estimated_time,departure_actual_time,site_specific_plan_id,operator,flight_type,route_text,initial_flight_rules,agreed.route.estimatedElapsedTime[0].elapsedTime,agreed.route.estimatedElapsedTime[0].location.region.airspaceType,…,agreed.route.expandedRoute.routePoint[22].point.location.pos,agreed.route.expandedRoute.routePoint[23].point.location.pos,agreed.route.expandedRoute.routePoint[24].point.location.pos,agreed.route.expandedRoute.routePoint[25].point.location.pos,agreed.route.expandedRoute.routePoint[26].point.location.pos,agreed.route.expandedRoute.routePoint[27].point.location.pos,agreed.route.expandedRoute.routePoint[28].point.location.pos,agreed.route.expandedRoute.routePoint[29].point.location.pos,agreed.route.expandedRoute.routePoint[30].point.location.pos,agreed.route.expandedRoute.routePoint[31].point.location.pos,agreed.route.expandedRoute.routePoint[32].point.location.pos,agreed.route.expandedRoute.routePoint[33].point.location.pos,agreed.route.expandedRoute.routePoint[34].point.location.pos,agreed.route.expandedRoute.routePoint[35].point.location.pos,agreed.route.expandedRoute.routePoint[36].point.location.pos,agreed.route.expandedRoute.routePoint[37].point.location.pos,agreed.route.expandedRoute.routePoint[38].point.location.pos,agreed.route.expandedRoute.routePoint[39].point.location.pos,agreed.route.expandedRoute.routePoint[40].estimatedTime,agreed.route.expandedRoute.routePoint[40].point.fix,agreed.route.expandedRoute.routePoint[40].point.distance,agreed.route.expandedRoute.routePoint[40].point.radial,agreed.route.expandedRoute.routePoint[41].estimatedTime,agreed.route.expandedRoute.routePoint[41].point.fix,agreed.route.expandedRoute.routePoint[42].estimatedTime,agreed.route.expandedRoute.routePoint[42].point.fix,agreed.route.expandedRoute.routePoint[39].point.distance,agreed.route.expandedRoute.routePoint[39].point.radial,requestedAltitude.block.above,requestedAltitude.block.below,enRoute.pointout.receivingUnit[3].unitIdentifier,enRoute.pointout.receivingUnit[3].sectorIdentifier,agreed.route.expandedRoute.routePoint[25].point.distance,agreed.route.expandedRoute.routePoint[25].point.radial,requestedAltitude.vfrOnTopPlus,assignedAltitude.vfrOnTopPlus,enRoute.alternateAerodrome[3].name
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""ZMA""","""FH""","""ATL""","""2026-03-01T15:10:51.550Z""","""KCLT""","""KFXE""",null,null,"""22X""","""TWY116""","""PROPOSED""","""3c9b5848-7b84-4964-a40d-116db6…","""KR546511Ak""","""211593346""","""us.fdps.2026-03-01T15:10:51Z.0…","""1""","""1510514455""","""15_10_51""",null,null,null,null,null,null,null,null,null,null,"""2026-03-02T12:02:00Z""",null,"""2528""","""SOLAIRUS AVIATION""","""GENERAL""","""KFXE.FRSBE2.STYMY.Q77.SHRKS..T…","""IFR""","""P0Y0M0DT0H26M0S""","""FIR""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,n


## intermediate SFDPS parquet, all columns
/Volumes/T2-SSD/planequery/data/intermediate/sfdps-logs/year=2026/month=03/day=01/sfdps-logs_2026_03_01.parquet
rows: 95


centre,source,system,timestamp,arrival_point,departure_point,position_speed_knots,position_latlon,computer_id,aircraft_identification,flight_status,gufi,flight_plan_id,sup_msg_seq_no,sup_fdps_gufi,sup_flight_plan_seq_no,sup_source_time_and_seq,sup_source_time,agreed.route.expandedRoute.routePoint[0].estimatedTime,agreed.route.expandedRoute.routePoint[0].point.fix,agreed.route.expandedRoute.routePoint[0].point.distance,agreed.route.expandedRoute.routePoint[0].point.radial,agreed.route.expandedRoute.routePoint[1].estimatedTime,agreed.route.expandedRoute.routePoint[1].point.fix,agreed.route.expandedRoute.routePoint[2].estimatedTime,agreed.route.expandedRoute.routePoint[2].point.fix,agreed.route.expandedRoute.routePoint[3].estimatedTime,agreed.route.expandedRoute.routePoint[3].point.fix,arrival_estimated_time,departure_actual_time,site_specific_plan_id,operator,flight_type,route_text,initial_flight_rules,agreed.route.estimatedElapsedTime[0].elapsedTime,agreed.route.estimatedElapsedTime[0].location.region.airspaceType,…,agreed.route.expandedRoute.routePoint[22].point.location.pos,agreed.route.expandedRoute.routePoint[23].point.location.pos,agreed.route.expandedRoute.routePoint[24].point.location.pos,agreed.route.expandedRoute.routePoint[25].point.location.pos,agreed.route.expandedRoute.routePoint[26].point.location.pos,agreed.route.expandedRoute.routePoint[27].point.location.pos,agreed.route.expandedRoute.routePoint[28].point.location.pos,agreed.route.expandedRoute.routePoint[29].point.location.pos,agreed.route.expandedRoute.routePoint[30].point.location.pos,agreed.route.expandedRoute.routePoint[31].point.location.pos,agreed.route.expandedRoute.routePoint[32].point.location.pos,agreed.route.expandedRoute.routePoint[33].point.location.pos,agreed.route.expandedRoute.routePoint[34].point.location.pos,agreed.route.expandedRoute.routePoint[35].point.location.pos,agreed.route.expandedRoute.routePoint[36].point.location.pos,agreed.route.expandedRoute.routePoint[37].point.location.pos,agreed.route.expandedRoute.routePoint[38].point.location.pos,agreed.route.expandedRoute.routePoint[39].point.location.pos,agreed.route.expandedRoute.routePoint[40].estimatedTime,agreed.route.expandedRoute.routePoint[40].point.fix,agreed.route.expandedRoute.routePoint[40].point.distance,agreed.route.expandedRoute.routePoint[40].point.radial,agreed.route.expandedRoute.routePoint[41].estimatedTime,agreed.route.expandedRoute.routePoint[41].point.fix,agreed.route.expandedRoute.routePoint[42].estimatedTime,agreed.route.expandedRoute.routePoint[42].point.fix,agreed.route.expandedRoute.routePoint[39].point.distance,agreed.route.expandedRoute.routePoint[39].point.radial,requestedAltitude.block.above,requestedAltitude.block.below,enRoute.pointout.receivingUnit[3].unitIdentifier,enRoute.pointout.receivingUnit[3].sectorIdentifier,agreed.route.expandedRoute.routePoint[25].point.distance,agreed.route.expandedRoute.routePoint[25].point.radial,requestedAltitude.vfrOnTopPlus,assignedAltitude.vfrOnTopPlus,enRoute.alternateAerodrome[3].name
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,…,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""ZMA""","""FH""","""ATL""","""2026-03-01T15:10:51.550Z""","""KCLT""","""KFXE""",null,null,"""22X""","""TWY116""","""PROPOSED""","""3c9b5848-7b84-4964-a40d-116db6…","""KR546511Ak""","""211593346""","""us.fdps.2026-03-01T15:10:51Z.0…","""1""","""1510514455""","""15_10_51""",null,null,null,null,null,null,null,null,null,null,"""2026-03-02T12:02:00Z""",null,"""2528""","""SOLAIRUS AVIATION""","""GENERAL""","""KFXE.FRSBE2.STYMY.Q77.SHRKS..T…","""IFR""","""P0Y0M0DT0H26M0S""","""FIR""",…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,n

missing /Users/jonahgoode/Documents/PlaneQuery/Code/planequery-flights-algorithm/data/intermediate/sfdps-logs/year=2026/month=03/day=01/sfdps-logs_2026_03_01.parquet
missing /Users/jonahgoode/Documents/PlaneQuery/Code/planequery-data/data/intermediate/sfdps-logs/year=2026/month=03/day=01/sfdps-logs_2026_03_01.parquet


In [7]:
pattern = re.compile("|".join(re.escape(v) for v in sorted(related_values, key=len, reverse=True)), re.IGNORECASE)

raw_hits = []
for raw_dir in raw_sfdps_dirs:
    print("\nraw dir", raw_dir)
    if not raw_dir.exists():
        print("missing")
        continue

    gz_files = sorted(p for p in raw_dir.iterdir() if p.suffix in {".gz", ".gzip"})
    print("gz files", len(gz_files))
    for gz_path in gz_files:
        with gzip.open(gz_path, "rt", encoding="utf-8", errors="replace") as fh:
            for line_number, line in enumerate(fh, start=1):
                if pattern.search(line):
                    raw_hits.append({
                        "path": str(gz_path),
                        "line_number": line_number,
                        "line": line.strip(),
                    })

df_raw_hits = pl.DataFrame(raw_hits) if raw_hits else pl.DataFrame({"path": [], "line_number": [], "line": []})
print("raw line hits", df_raw_hits.height)
display(df_raw_hits)


raw dir /Volumes/T2-SSD/planequery/data/raw/sfdps-logs/year=2026/month=03/day=01
gz files 470


KeyboardInterrupt: 

In [ ]:
timeline_frames = []

for df in derived_matches:
    cols = [c for c in ["icao", "callsign", "registration", "takeoff_time", "takeoff_airport_ident", "landing_time", "landing_airport_ident", "aircraft_type", "owner", "aircraft_description"] if c in df.columns]
    timeline_frames.append(df.select(cols).with_columns(pl.lit("derived_sfdps").alias("source")))

for df in algorithm_matches:
    cols = [c for c in ["icao", "callsign", "registration", "takeoff_time", "takeoff_airport_ident", "landing_time", "landing_airport_ident", "first_message_time", "last_message_time", "flight_id"] if c in df.columns]
    timeline_frames.append(df.select(cols).with_columns(pl.lit("algorithm").alias("source")))

if timeline_frames:
    timeline = pl.concat(timeline_frames, how="diagonal_relaxed")
    sort_cols = [c for c in ["takeoff_time", "landing_time", "source"] if c in timeline.columns]
    if sort_cols:
        timeline = timeline.sort(sort_cols)
    display(timeline)
else:
    print("No timeline rows found in derived or algorithm outputs.")